# 🎤 Voice-to-Action Campus Assistant
### From speech → intent → confirmed action — free-tier OpenRouter + one small local library

**Session goal:** build an agent that takes a *spoken* instruction ("book me the seminar hall for
Friday 3pm"), transcribes it, figures out what you actually want, and only acts once it's
confident — asking a follow-up question instead of guessing when it isn't.

**Reasoning and transcription run on ONE OpenRouter API key**, using only **free-tier (`:free`)
models** — no payment method needed. Two things worth knowing up front, since they shape how this
lab is built:
- OpenRouter does **not** have separate `/audio/speech` or `/audio/transcriptions` endpoints the
  way OpenAI's API does. Audio is handled as an *input modality on the normal chat completions
  endpoint* — you send base64-encoded audio inside a chat message, to a model that supports audio
  input, and ask it to transcribe. We use that for Step 2.
- There is no free (or even generally available) **text-to-speech** model on OpenRouter today. So
  for Step 1 (generating sample voice clips to work with), we use a tiny free local library
  (`gTTS`) instead — it needs no API key and no billing, but it is not an OpenRouter call. Once we
  have audio in hand, everything downstream (transcription, intent extraction, confidence
  gating, follow-up questions) goes through OpenRouter.

**What we build, step by step:**

| Step | What we add | Concept |
|---|---|---|
| 1 | Generate sample voice clips locally (gTTS) | Getting audio input to work with, offline |
| 2 | Transcribe audio → raw text via OpenRouter | Speech-to-text via a multimodal chat model — and why the transcript itself can be messy |
| 3 | Naive baseline: transcript → booking, no checks | Motivating failure case |
| 4 | Structured intent extraction + self-reported confidence | Structured output under ambiguity |
| 5 | Confidence-gated branching: act vs. ask a follow-up | The core new concept |
| 6 | Deterministic availability check + confirm-before-book | Read/write split + self-confirm HITL |
| 7 | Full assembled agent + class exercise | Putting it together |


## 🔑 Step 0 — OpenRouter setup

1. Sign up at **[openrouter.ai](https://openrouter.ai)**.
2. Create a key at **[openrouter.ai/keys](https://openrouter.ai/keys)**.
3. In Colab: **🔑 key icon (left sidebar) → Secrets → add `OPENROUTER_API_KEY`** → toggle
   "Notebook access" ON.

**⚠️ Free-tier models only:** same rule as the Budget Agent lab — stick to model ids ending in
`:free` (check [openrouter.ai/models](https://openrouter.ai/models?max_price=0) since the list
changes). This lab needs a free model that supports **both tool calling and audio input**, since
we use it for structured extraction *and* transcription. As of writing:
- `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free` (default below — audio+image+video input, tools)
- `thinkingmachines/inkling:free` / `thinkingmachines/inkling-small:free` (audio+image input, tools)

**Rate limits:** with no credits ever purchased, `:free` models are capped at 20 requests/minute
and 50 requests/day per key. This lab calls the model for every transcription *and* every
extraction, so pace yourself during the class exercises.

> One OpenRouter key, two things it's used for today: chat completions for reasoning (structured
> extraction, follow-up questions) and chat completions with audio input for transcription. Text-
> to-speech for the demo clips is handled separately, locally, with no key — see Step 1.


In [ ]:
!pip install openai gtts --quiet
print("✅ Installed.")


In [ ]:
import os
from google.colab import userdata

try:
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    if not OPENROUTER_API_KEY:
        raise ValueError("empty")
except Exception:
    OPENROUTER_API_KEY = input("Paste your OpenRouter API key: ").strip()

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
print("✅ Key loaded." if OPENROUTER_API_KEY else "❌ No key found.")


In [ ]:
from openai import OpenAI
import json

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# Free-tier model (":free" suffix = $0 cost) that supports BOTH tool calling and audio input —
# we use the same model for transcription (Step 2) and all reasoning (Steps 3-5). Swap for
# another ":free" audio+tools model to compare — see Step 0 for current options.
MODEL = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"

def ask(messages, tools=None, tool_choice=None, **kwargs):
    """Thin wrapper around chat completions."""
    return client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools, tool_choice=tool_choice, **kwargs,
    )

print("✅ Client ready.")


---
## Step 1 — Generate sample voice clips (text-to-speech, done locally)

We need voice input to work with. Rather than depending on 30 laptops' microphones working
reliably mid-session, we'll **generate** three sample clips — one clean instruction, one
filler-heavy-but-complete instruction, and one genuinely ambiguous one.
(A live mic-recording cell is included later as a bonus if you want to try your own voice.)

**Note:** this step uses `gTTS`, a small free local library — not OpenRouter. OpenRouter doesn't
have a free text-to-speech model today, so this is the one part of the notebook that isn't an
OpenRouter call. Everything from Step 2 onward (transcription, reasoning) goes through your
OpenRouter key and the `MODEL` set in Step 0.


In [ ]:
import pathlib
from gtts import gTTS

CLIPS_DIR = pathlib.Path("clips")
CLIPS_DIR.mkdir(exist_ok=True)

SAMPLE_TEXTS = {
    "clear": "Please book the seminar hall for Friday at 3 PM.",
    "filler_heavy": "So, uh, I was thinking, could you maybe, like, book the seminar hall, "
                     "uh, for Friday, 3 pm, if that's cool?",
    "ambiguous": "Um, yeah, book that place for the thing tomorrow, sometime in the afternoon I guess.",
}

def synthesize_speech(text: str, out_path: pathlib.Path) -> pathlib.Path:
    """Text -> audio file, using the free local gTTS library (no API key, not OpenRouter)."""
    gTTS(text=text, lang="en").save(str(out_path))
    return out_path

clip_paths = {}
for label, text in SAMPLE_TEXTS.items():
    path = CLIPS_DIR / f"{label}.mp3"
    synthesize_speech(text, path)
    clip_paths[label] = path
    print(f"✅ Generated {label} -> {path}  (source text: \"{text}\")")


In [ ]:
# Listen to one of them right here in Colab
from IPython.display import Audio, display

print("Playing: clear instruction")
display(Audio(str(clip_paths["clear"])))


---
## Step 2 — Speech-to-text: transcribe the clips

Now the reverse direction: audio → text. OpenRouter doesn't have a dedicated transcription
endpoint — instead, we send the audio **as part of a normal chat completion**, using the
`input_audio` content type, to our audio-capable `MODEL`, and simply ask it to transcribe. Same
`client.chat.completions.create` call as everywhere else in this notebook, just with a different
content shape.


In [ ]:
import base64

def transcribe(path: pathlib.Path) -> str:
    """Audio file -> raw transcript text, via a chat completion with audio input."""
    audio_b64 = base64.b64encode(path.read_bytes()).decode("utf-8")
    audio_format = path.suffix.lstrip(".")  # e.g. "mp3"
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": "Transcribe this audio exactly, word for word. "
                                          "Reply with ONLY the transcript text, nothing else."},
                {"type": "input_audio", "input_audio": {"data": audio_b64, "format": audio_format}},
            ],
        }],
    )
    return resp.choices[0].message.content.strip()

transcripts = {label: transcribe(path) for label, path in clip_paths.items()}
for label, text in transcripts.items():
    print(f"[{label}] -> {text!r}")



### 🔍 Discuss in class

Even with clean, synthetic audio (no real background noise), notice:
- The **filler-heavy** clip's transcript still carries "um," "uh," "like" — a real STT transcript
  is *not* clean input, even when the audio itself is clear.
- The **ambiguous** clip's transcript is grammatically fine but semantically underspecified —
  "that place," "the thing," "sometime in the afternoon" carry no bookable specifics.

**Key lesson:** the agent's first real job is often just "how much can I actually extract from
this text?" — not "did I understand every word?" Those are different questions.



---
## Step 3 — Naive baseline: transcript straight to a booking decision

Let's do what most people would build first: hand the raw transcript to the model and ask it to
just book something. Watch what happens with the ambiguous clip.


In [ ]:
naive_prompt = (
    "A user said this to a room-booking assistant: \"" + transcripts["ambiguous"] + "\"\n"
    "Book the room they're asking for. Respond with what you booked."
)
response = ask([{"role": "user", "content": naive_prompt}])
print(response.choices[0].message.content)



### 🔍 What just happened

The model almost certainly **guessed** a specific room and time rather than admitting it doesn't
know — the same sycophancy/overconfidence failure from the Budget Agent's naive baseline, now in
a voice context. In a real deployment this means: **wrong room gets booked, silently, with total
confidence.** This is the motivating failure case for the rest of the lab.



---
## Step 4 — Structured intent extraction with self-reported confidence

Same forced-JSON-schema pattern as the Budget Agent, but now the schema requires the model to
report **how confident it is**, and to explicitly list **what's missing**, instead of silently
filling gaps with a guess.


In [ ]:
intent_schema = {
    "type": "function",
    "function": {
        "name": "extract_booking_intent",
        "description": "Extract a room-booking request from a (possibly messy) voice transcript.",
        "parameters": {
            "type": "object",
            "properties": {
                "room": {"type": "string", "description": "Room name, or empty string if not stated"},
                "day": {"type": "string", "description": "Day of week or date, or empty string if not stated"},
                "time": {"type": "string", "description": "Time, or empty string if not stated"},
                "confidence": {
                    "type": "number",
                    "description": "0.0-1.0: how confident are you that room/day/time are all correctly and unambiguously specified?",
                },
                "missing_or_ambiguous_fields": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Which fields are missing, vague, or ambiguous (e.g. 'room', 'time')",
                },
            },
            "required": ["room", "day", "time", "confidence", "missing_or_ambiguous_fields"],
        },
    },
}

def extract_intent(transcript: str) -> dict:
    resp = ask(
        [{"role": "user", "content": f"Transcript: \"{transcript}\""}],
        tools=[intent_schema],
        tool_choice={"type": "function", "function": {"name": "extract_booking_intent"}},
    )
    call = resp.choices[0].message.tool_calls[0]
    return json.loads(call.function.arguments)

for label in ["clear", "filler_heavy", "ambiguous"]:
    intent = extract_intent(transcripts[label])
    print(f"\n[{label}]")
    print(json.dumps(intent, indent=2))



### 🔍 Compare the three

You should see the **clear** and **filler_heavy** clips land on high confidence with all fields
filled (filler words didn't actually remove information) — while **ambiguous** correctly reports
low confidence with `missing_or_ambiguous_fields` populated. This is the branch point for Step 5.



---
## Step 5 — Confidence-gated branching: act vs. ask a follow-up

This is the core new concept. **The confidence check happens in your code, not inside the
model's own free-text answer** — same principle as the Budget Agent's feasibility check being a
real Python comparison, not something you trust the model to self-assess in prose.

If confidence is high enough *and* nothing is missing → proceed to booking.
Otherwise → generate a targeted follow-up question and wait for more input, instead of guessing.


In [ ]:
CONFIDENCE_THRESHOLD = 0.7

def needs_clarification(intent: dict) -> bool:
    return intent["confidence"] < CONFIDENCE_THRESHOLD or len(intent["missing_or_ambiguous_fields"]) > 0

def generate_followup_question(intent: dict) -> str:
    missing = intent["missing_or_ambiguous_fields"] or ["details"]
    prompt = (
        f"A user's booking request was unclear on: {missing}. "
        "Write ONE short, friendly follow-up question to ask them, to resolve just that ambiguity."
    )
    resp = ask([{"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip()

for label in ["clear", "ambiguous"]:
    intent = extract_intent(transcripts[label])
    print(f"\n[{label}] confidence={intent['confidence']}")
    if needs_clarification(intent):
        question = generate_followup_question(intent)
        print("🤔 Needs clarification. Agent asks:", question)
    else:
        print("✅ Confident enough to proceed to booking.")



### 🔍 Try breaking it on purpose

Set `CONFIDENCE_THRESHOLD = 0.99` and re-run — now even the **clear** clip should trigger a
follow-up question, because almost nothing clears a 99% bar. Then set it to `0.1` and watch the
**ambiguous** clip go straight to booking anyway.

This is the exact demo to run live: **skip the confidence check entirely (set threshold to `0`)**
and show the agent proceeding to book a guessed room/time from the ambiguous transcript with total
confidence in its own words — even though the underlying intent extraction told you it wasn't sure.



---
## Step 6 — Deterministic availability check + confirm-before-book

Booking a room is a **side effect** — once booked, someone else can't book it. So even a
high-confidence request doesn't get to skip straight to booking:

1. **Availability check** — a deterministic lookup (never the model's judgment; same pattern as
   `estimate_trip_cost` in the Budget Agent).
2. **Self-confirm step** — the user (who spoke the request) confirms the *interpreted* request
   before it becomes real. This is a lighter-weight cousin of the GitHub Triager's human-approval
   gate: there, a *third party* approves someone else's proposed action; here, the *same person*
   confirms their own noisy input was understood correctly. Same underlying principle — a
   side-effecting action always pauses for a human — applied at a different point in the pipeline.
3. **Idempotency check** — don't double-book if this exact request was already booked.


In [ ]:
# A tiny simulated room calendar: (room, day, time) -> booked_by (or None if free)
ROOM_CALENDAR = {
    ("seminar hall", "friday", "3 pm"): None,
    ("seminar hall", "friday", "5 pm"): "Robotics Club",
    ("conference room", "friday", "3 pm"): None,
}

def normalize(value: str) -> str:
    return value.strip().lower()

def check_availability(room: str, day: str, time: str) -> dict:
    """Deterministic tool — never delegate this judgment to the model."""
    key = (normalize(room), normalize(day), normalize(time))
    if key not in ROOM_CALENDAR:
        return {"known_slot": False, "available": False, "reason": "Slot not found in calendar"}
    occupied_by = ROOM_CALENDAR[key]
    return {"known_slot": True, "available": occupied_by is None, "reason": occupied_by}

def book_room(room: str, day: str, time: str, requester: str = "You") -> dict:
    """Side-effecting action — only ever called AFTER a human confirmation, see Step 7."""
    key = (normalize(room), normalize(day), normalize(time))
    if ROOM_CALENDAR.get(key) is not None:
        return {"success": False, "reason": f"Already booked by {ROOM_CALENDAR[key]} (idempotency check caught this)"}
    ROOM_CALENDAR[key] = requester
    return {"success": True, "reason": f"Booked {room} on {day} at {time} for {requester}"}

# Quick tests
print(check_availability("seminar hall", "friday", "3 pm"))
print(check_availability("seminar hall", "friday", "5 pm"))



---
## Step 7 — The full Voice-to-Action Agent

Wiring Steps 2–6 into one function: audio in → confident, confirmed booking (or an honest
clarification request) out.


In [ ]:
def voice_to_action_agent(audio_path: pathlib.Path, auto_confirm: bool = False) -> dict:
    print("🎧 Step A — Transcribing audio...")
    transcript = transcribe(audio_path)
    print("Transcript:", transcript)

    print("\n🧠 Step B — Extracting structured intent...")
    intent = extract_intent(transcript)
    print(json.dumps(intent, indent=2))

    print("\n🚦 Step C — Confidence gate...")
    if needs_clarification(intent):
        question = generate_followup_question(intent)
        print("🤔 Not confident enough. Agent would ask:", question)
        return {"status": "needs_clarification", "question": question, "intent": intent}

    room, day, time = intent["room"], intent["day"], intent["time"]
    print(f"✅ Confident. Interpreted request: {room} / {day} / {time}")

    print("\n📅 Step D — Checking availability (deterministic)...")
    avail = check_availability(room, day, time)
    print(avail)
    if not avail["available"]:
        return {"status": "unavailable", "detail": avail, "intent": intent}

    print("\n🙋 Step E — Confirmation (self-confirm HITL gate)...")
    if auto_confirm:
        confirmed = True
        print(f"(auto_confirm=True) Confirming: {room}, {day} at {time}")
    else:
        answer = input(f"Book {room} on {day} at {time}? [y/n]: ").strip().lower()
        confirmed = answer == "y"

    if not confirmed:
        return {"status": "cancelled_by_user", "intent": intent}

    print("\n📝 Step F — Booking (side-effecting action, only now that it's confirmed)...")
    result = book_room(room, day, time)
    print(result)
    return {"status": "booked" if result["success"] else "booking_failed", "detail": result, "intent": intent}


In [ ]:
# Try the clear clip first — should sail through to the confirmation prompt
result = voice_to_action_agent(clip_paths["clear"])
print("\nFINAL:", result["status"])


In [ ]:
# Now try the ambiguous clip — should stop at clarification instead of guessing
result = voice_to_action_agent(clip_paths["ambiguous"])
print("\nFINAL:", result["status"])



---
## 🎙️ Bonus (optional): record your own voice live in Colab

Mic access in Colab depends on your browser/OS permissions, so this is a stretch-goal cell, not
the main path — if it doesn't work, keep using the generated clips above.


In [ ]:
# OPTIONAL — live mic recording in Colab. If this cell fails or hangs, skip it and use the
# generated sample clips instead (this is expected on some browsers/OS/mic-permission setups).
from IPython.display import Javascript
from base64 import b64decode
import uuid

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
var recorder;
async function record(seconds) {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    recorder = new MediaRecorder(stream);
    const chunks = [];
    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.start();
    await sleep(seconds * 1000);
    recorder.stop();
    await new Promise(resolve => recorder.onstop = resolve);
    const blob = new Blob(chunks);
    const buffer = await blob.arrayBuffer();
    const bytes = new Uint8Array(buffer);
    let binary = "";
    for (let i = 0; i < bytes.byteLength; i++) binary += String.fromCharCode(bytes[i]);
    return btoa(binary);
}
"""

def record_audio(seconds: int = 5) -> pathlib.Path:
    from google.colab import output
    display(Javascript(RECORD_JS))
    b64_audio = output.eval_js(f"record({seconds})")
    out_path = CLIPS_DIR / f"mic_{uuid.uuid4().hex[:8]}.webm"
    out_path.write_bytes(b64decode(b64_audio))
    return out_path

print("Run: my_clip = record_audio(5)   # then speak a booking request for 5 seconds")


In [ ]:
# Uncomment to try live recording:
# my_clip = record_audio(5)
# result = voice_to_action_agent(my_clip)
# print(result["status"])



---
## 🧪 Try it yourself (class exercise)

1. Run `voice_to_action_agent(clip_paths["filler_heavy"])` — confirm filler words alone don't
   trigger a false clarification (the info is all there, just wrapped in "um"s).
2. Generate your own clip: `synthesize_speech("your text here", CLIPS_DIR / "mine.mp3")`, then run
   the full agent on it. Try wording that's deliberately missing the room OR the time OR the day.
3. Set `CONFIDENCE_THRESHOLD = 0` and re-run the ambiguous clip — watch it skip straight past a
   clarification it should have asked for, and book something it was never actually sure about.
4. Try booking the same slot twice in a row (`book_room("seminar hall", "friday", "3 pm")` called
   twice) — confirm the idempotency check stops the second one.
5. **Stretch:** add a fourth ROOM_CALENDAR entry and test a request for a slot that doesn't exist
   at all — confirm `check_availability` reports `known_slot: False` rather than the model guessing.

## 🧵 Recap — what you actually built

| Layer | What it means in production systems |
|---|---|
| TTS/STT (Steps 1-2) | A multimodal pipeline stage that's lossy *before* any reasoning starts |
| Structured intent + confidence (Step 4) | Don't let the model silently guess — make it self-report uncertainty as data |
| Confidence gate (Step 5) | The branch between "act" and "ask" must be a real check in your code, not vibes in the model's prose |
| Availability + confirm + idempotency (Step 6) | Side-effecting actions get a human checkpoint and a re-run-safe guard, even for low-stakes actions |

**The one-line takeaway:**
> Every stage of a multimodal pipeline can silently lose information — the job of the agent isn't
> just "understand the words," it's "know when it doesn't know," and only act once it does.
